# AOD Comparison: Satellite vs AERONET Ground Truth — Hanoi

## Sources
| Source | Type | Level | AOD wavelength | Temporal |
|--------|------|-------|---------------|----------|
| **AERONET** Nghia Do | Ground | L1.5 | 500nm → interpolated 550nm | Multi-daily (~5min) |
| **VIIRS** NOAA-20 | Polar orbit | L2 (nearest pixel) | 550nm | ~1 overpass/day (04-06 UTC) |
| **VIIRS** NOAA-20 | Polar orbit | D3 (daily 1° grid) | 550nm | Daily composite |
| **Himawari-8/9** | Geostationary | L2 (10-min) | 550nm (AOT) | 10-min intervals |
| **Himawari-8/9** | Geostationary | L3 (hourly) | 550nm (AOT_Merged/L2_Mean) | Hourly |
| **MODIS** MCD19A2 | Polar orbit | L2 (MAIAC) | 550nm | ~1-2 overpasses/day |

## Comparison Constraints
- **L2 group**: VIIRS L2 ↔ Himawari L2 ↔ MODIS ↔ AERONET (time-matched)
- **L3 group**: VIIRS D3 ↔ Himawari L3 ↔ AERONET (daily aggregated)
- **Never** compare L2 with D3/L3 directly

## 3 Hanoi Stations
556 Nguyễn Văn Cừ · Công viên Nhân Chính · ĐHBK cổng Parabol Giải Phóng

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (14, 6), "axes.titlesize": 13, "font.size": 11, "figure.dpi": 120})

# Station name mapping across datasets
STATION_MAP = {
    "NVC": {
        "aeronet": "NGHIA_DO",  # single AERONET site for all 3 stations (closest)
        "viirs": "HN: 556 Nguyễn Văn Cừ",
        "him_csv": "Hà Nội: 556 Nguyễn Văn Cừ",
        "modis": "Hà Nội: 556 Nguyễn Văn Cừ (KK)",
        "label": "556 Nguyễn Văn Cừ",
    },
    "NhanChinh": {
        "aeronet": "NGHIA_DO",
        "viirs": "HN: CV Nhân Chính",
        "him_csv": "Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến",
        "modis": "Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến (KK)",
        "label": "CV Nhân Chính",
    },
    "DHBK": {
        "aeronet": "NGHIA_DO",
        "viirs": "HN: ĐHBK Giải Phóng",
        "him_csv": "Hà Nội: ĐHBK cổng Parabol đường Giải Phóng",
        "modis": "Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK)",
        "label": "ĐHBK Giải Phóng",
    },
}

COLORS = {
    "AERONET": "#2196F3", "VIIRS_L2": "#4CAF50", "VIIRS_D3": "#8BC34A",
    "Himawari_L2": "#FFD54F", "Himawari_L3": "#FF9800", "MODIS": "#9C27B0",
}

NODATA = -9999.0
TIME_WINDOW_MIN = 30  # ±30 min for L2 time matching (standard in AOD validation)

print("Setup complete.")

## 1. Load All Data Sources

In [ ]:
# ── 1.1 AERONET (ground truth) ──
# Interpolate AOD_500 → AOD_550 using Angstrom Exponent:
#   AOD_550 = AOD_500 * (550/500)^(-AE_440-675)
aeronet_path = "/home/slow_data/Air_Quality/AERONET/NGHIA_DO/aeronet_NGHIA_DO_L15_2022-01-01_2026-04-01.csv"
df_aeronet = pd.read_csv(aeronet_path)
df_aeronet["datetime"] = pd.to_datetime(df_aeronet["datetime"])
df_aeronet["AOD_500"] = pd.to_numeric(df_aeronet["AOD_500nm"], errors="coerce")

# Use first non-polar AE column for interpolation
ae_cols = [c for c in df_aeronet.columns if c == "440-675_Angstrom_Exponent"]
if not ae_cols:
    ae_cols = [c for c in df_aeronet.columns if "440-675_Angstrom_Exponent" in c and "Polar" not in c]
df_aeronet["AE"] = pd.to_numeric(df_aeronet[ae_cols[0]], errors="coerce")
df_aeronet["AOD_550"] = df_aeronet["AOD_500"] * (550.0 / 500.0) ** (-df_aeronet["AE"])

# Also compute AOD at 551nm directly if available (for cross-check)
if "AOD_551nm" in df_aeronet.columns:
    df_aeronet["AOD_551_direct"] = pd.to_numeric(df_aeronet["AOD_551nm"], errors="coerce")

df_aeronet = df_aeronet.dropna(subset=["AOD_550"]).reset_index(drop=True)
df_aeronet["date"] = df_aeronet["datetime"].dt.date

print(f"AERONET: {len(df_aeronet):,} records, {df_aeronet['datetime'].min().date()} → {df_aeronet['datetime'].max().date()}")
print(f"  AOD_550: mean={df_aeronet['AOD_550'].mean():.3f}, range=[{df_aeronet['AOD_550'].min():.3f}, {df_aeronet['AOD_550'].max():.3f}]")

In [ ]:
# ── 1.2 VIIRS L2 (nearest pixel) + D3 (daily 1° grid) ──
# VIIRS extracts AOD at 550nm
df_viirs_l2 = pd.read_csv(
    "/home/slow_data/Air_Quality/VIIRS/output/viirs_aod_L2_NOAA20_nearest_filtered.csv",
    parse_dates=["datetime"],
    usecols=["datetime", "station", "station_lat", "station_lon", "aod", "qa", "dist_km"]
)
df_viirs_l2["date"] = df_viirs_l2["datetime"].dt.date
print(f"VIIRS L2: {len(df_viirs_l2):,} records, hours UTC={sorted(df_viirs_l2['datetime'].dt.hour.unique())}")
print(f"  stations: {df_viirs_l2['station'].unique().tolist()}")

df_viirs_d3 = pd.read_csv(
    "/home/slow_data/Air_Quality/VIIRS/output/viirs_aod_D3_NOAA20.csv",
    parse_dates=["datetime"],
    usecols=["datetime", "station", "station_lat", "station_lon", "aod_mean", "aod_count"]
)
df_viirs_d3["date"] = df_viirs_d3["datetime"].dt.date
print(f"VIIRS D3: {len(df_viirs_d3):,} records, date range: {df_viirs_d3['datetime'].min().date()} → {df_viirs_d3['datetime'].max().date()}")

In [ ]:
# ── 1.3 Himawari L2 (10-min) + L3 (hourly) from station CSV extracts ──
him_l2_dir = "/home/slow_data/Air_Quality/Himawari/station_aod/envisoft_stations"
him_l3_dir = "/home/slow_data/Air_Quality/Himawari/station_aod/L3_envisoft_stations"

him_station_files = {
    "NVC":        "Hà Nội: 556 Nguyễn Văn Cừ.csv",
    "NhanChinh":  "Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến.csv",
    "DHBK":       "Hà Nội: ĐHBK cổng Parabol đường Giải Phóng.csv",
}

def load_himawari_csv(base_dir, filenames, level):
    """Load Himawari station CSV files (L2 or L3)."""
    dfs = []
    for skey, fname in filenames.items():
        fpath = os.path.join(base_dir, fname)
        if not os.path.exists(fpath):
            print(f"  SKIP (not found): {fpath}")
            continue
        d = pd.read_csv(fpath)
        d["timestamp"] = d["timestamp"].astype(str)
        d["datetime"] = pd.to_datetime(d["timestamp"], format="%Y%m%d_%H%M", errors="coerce")
        d["station"] = STATION_MAP[skey]["him_csv"]

        if level == "L2":
            # L2: AOT column, filter by QA (valid QA < 4096 per Himawari convention)
            d["aod"] = pd.to_numeric(d["AOT"], errors="coerce")
            d.loc[d["aod"] <= 0, "aod"] = np.nan
        else:
            # L3: AOT_Merged + AOT_L2_Mean; replace -9999 with NaN
            for col in ["AOT_Merged", "AOT_Pure", "AOT_L2_Mean", "AOT_L2_SDV"]:
                if col in d.columns:
                    d[col] = pd.to_numeric(d[col], errors="coerce")
                    d.loc[d[col] <= NODATA + 1, col] = np.nan
            d["aod"] = d["AOT_Merged"].where(d["AOT_Merged"].notna(), d.get("AOT_L2_Mean"))

        d = d.dropna(subset=["datetime"]).reset_index(drop=True)
        d["date"] = d["datetime"].dt.date
        dfs.append(d)

    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)

print("Loading Himawari L2 (10-min)...")
df_him_l2 = load_himawari_csv(him_l2_dir, him_station_files, "L2")
df_him_l2_valid = df_him_l2.dropna(subset=["aod"])
print(f"  L2: {len(df_him_l2):,} total, {len(df_him_l2_valid):,} with valid AOD")

print("Loading Himawari L3 (hourly)...")
df_him_l3 = load_himawari_csv(him_l3_dir, him_station_files, "L3")
df_him_l3_valid = df_him_l3.dropna(subset=["aod"])
print(f"  L3: {len(df_him_l3):,} total, {len(df_him_l3_valid):,} with valid AOD")

In [ ]:
# ── 1.4 MODIS MCD19A2 (L2 — MAIAC AOD at 550nm) ──
modis_files = {
    "NVC":        "/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod/MODIS_AQI_28560877461938780203765592307.csv",
    "NhanChinh":  "/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod/MODIS_AQI_31390908889087377344742439468.csv",
    "DHBK":       "/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod/MODIS_AQI_31390903576425084107499649578.csv",
}

dfs_modis = []
for skey, fp in modis_files.items():
    if not os.path.exists(fp):
        print(f"  SKIP (not found): {fp}")
        continue
    d = pd.read_csv(fp)
    d["datetime"] = pd.to_datetime(d["timestamp"], errors="coerce")
    d["station"] = STATION_MAP[skey]["viirs"]  # use VIIRS station naming for consistency
    d["aod"] = pd.to_numeric(d["Optical_Depth_055"], errors="coerce")
    d["date"] = d["datetime"].dt.date
    dfs_modis.append(d[["datetime", "date", "station", "aod"]].dropna(subset=["aod"]))

df_modis = pd.concat(dfs_modis, ignore_index=True) if dfs_modis else pd.DataFrame()
print(f"MODIS: {len(df_modis):,} records, hours UTC={sorted(df_modis['datetime'].dt.hour.unique()) if len(df_modis) else 'N/A'}")

## 2. Time Matching Functions

Standard AOD validation methodology (Ichoku et al. 2002, Levy et al. 2010):
- **L2 matching**: For each satellite overpass, average AERONET observations within ±30 min window
- **L3/D3 matching**: Compare daily means (AERONET full-day mean vs satellite daily product)

In [ ]:
def match_l2_to_aeronet(df_sat, df_aer, window_min=TIME_WINDOW_MIN):
    """
    For each satellite L2 observation, find AERONET mean AOD within ±window_min.
    Returns DataFrame with columns: datetime, sat_aod, aer_aod, station.
    """
    window = pd.Timedelta(minutes=window_min)
    aer = df_aer[["datetime", "AOD_550"]].copy().set_index("datetime").sort_index()
    records = []

    for _, row in df_sat.iterrows():
        t_sat = row["datetime"]
        t0, t1 = t_sat - window, t_sat + window
        aer_window = aer.loc[t0:t1, "AOD_550"]
        if len(aer_window) >= 1:
            records.append({
                "datetime": t_sat,
                "date": t_sat.date(),
                "sat_aod": row["aod"],
                "aer_aod": aer_window.mean(),
                "aer_std": aer_window.std() if len(aer_window) > 1 else 0,
                "aer_n": len(aer_window),
                "station": row.get("station", ""),
            })
    return pd.DataFrame(records)


def match_daily_to_aeronet(df_sat, df_aer, sat_aod_col="aod"):
    """
    Match satellite daily product with AERONET daily mean.
    Aggregates both to daily level, then inner-joins on date.
    """
    aer_daily = df_aer.groupby("date")["AOD_550"].agg(["mean", "std", "count"]).reset_index()
    aer_daily.columns = ["date", "aer_aod", "aer_std", "aer_n"]

    sat_daily = df_sat.groupby("date")[sat_aod_col].mean().reset_index()
    sat_daily.columns = ["date", "sat_aod"]

    merged = aer_daily.merge(sat_daily, on="date").dropna()
    merged["datetime"] = pd.to_datetime(merged["date"])
    return merged


def compute_metrics(aer, sat):
    """Compute standard AOD validation metrics."""
    mask = np.isfinite(aer) & np.isfinite(sat)
    x, y = np.array(aer)[mask], np.array(sat)[mask]
    n = len(x)
    if n < 3:
        return {"N": n}

    slope, intercept, r_value, p_value, _ = stats.linregress(x, y)
    rmse = np.sqrt(np.mean((y - x) ** 2))
    mae = np.mean(np.abs(y - x))
    bias = np.mean(y - x)
    # Expected Error envelope: ±(0.05 + 0.15*AOD_AERONET) — standard for MODIS/VIIRS
    within_ee = np.mean(np.abs(y - x) <= (0.05 + 0.15 * x)) * 100

    return {
        "N": n, "R": r_value, "R²": r_value**2,
        "RMSE": rmse, "MAE": mae, "Bias": bias,
        "Slope": slope, "Intercept": intercept,
        "%EE": within_ee, "p_value": p_value,
    }

print("Time matching functions defined.")

In [ ]:
# ── 2.1 Perform L2-level time matching (±30 min) ──
print("Matching VIIRS L2 to AERONET (±30 min)...")
matched_viirs_l2 = match_l2_to_aeronet(df_viirs_l2.rename(columns={"aod": "aod"}), df_aeronet)
print(f"  VIIRS L2: {len(matched_viirs_l2)} matched pairs")

print("Matching Himawari L2 to AERONET (±30 min)...")
if len(df_him_l2_valid) > 0:
    matched_him_l2 = match_l2_to_aeronet(df_him_l2_valid, df_aeronet)
    print(f"  Himawari L2: {len(matched_him_l2)} matched pairs")
else:
    matched_him_l2 = pd.DataFrame()
    print("  Himawari L2: no valid AOD data")

print("Matching MODIS to AERONET (±30 min)...")
if len(df_modis) > 0:
    matched_modis = match_l2_to_aeronet(df_modis, df_aeronet)
    print(f"  MODIS: {len(matched_modis)} matched pairs")
else:
    matched_modis = pd.DataFrame()
    print("  MODIS: no data")

# ── 2.2 Perform L3/D3-level daily matching ──
print("\nMatching VIIRS D3 to AERONET (daily)...")
matched_viirs_d3 = match_daily_to_aeronet(df_viirs_d3, df_aeronet, sat_aod_col="aod_mean")
print(f"  VIIRS D3: {len(matched_viirs_d3)} matched days")

print("Matching Himawari L3 to AERONET (daily)...")
if len(df_him_l3_valid) > 0:
    matched_him_l3 = match_daily_to_aeronet(df_him_l3_valid, df_aeronet, sat_aod_col="aod")
    print(f"  Himawari L3: {len(matched_him_l3)} matched days")
else:
    matched_him_l3 = pd.DataFrame()
    print("  Himawari L3: no valid AOD data")

## 3. Data Overview & Distributions

In [ ]:
# ── 3.1 Temporal coverage: monthly record counts + mean AOD per source ──
datasets_info = [
    ("AERONET", df_aeronet, "AOD_550"),
    ("VIIRS L2", df_viirs_l2, "aod"),
    ("VIIRS D3", df_viirs_d3, "aod_mean"),
    ("MODIS", df_modis, "aod"),
]
# Add Himawari only if data exists
if len(df_him_l2_valid) > 0:
    datasets_info.append(("Himawari L2", df_him_l2_valid, "aod"))
if len(df_him_l3_valid) > 0:
    datasets_info.append(("Himawari L3", df_him_l3_valid, "aod"))

n_ds = len(datasets_info)
ncols = min(3, n_ds)
nrows = (n_ds + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).flatten()

for ax, (name, df, col) in zip(axes, datasets_info):
    tmp = df.copy()
    tmp["ym"] = pd.to_datetime(tmp["datetime"]).dt.to_period("M")
    m = tmp.groupby("ym").agg(n=(col, "count"), mu=(col, "mean")).reset_index()
    m["ym_s"] = m["ym"].astype(str)

    ax2 = ax.twinx()
    ax.bar(range(len(m)), m["n"], color=COLORS.get(name.replace(" ", "_"), "gray"), alpha=0.5)
    ax2.plot(range(len(m)), m["mu"], "o-", color="red", ms=3, lw=1)
    ax.set_title(f"{name} — monthly")
    ax.set_ylabel("Count")
    ax2.set_ylabel("Mean AOD", color="red")
    ticks = list(range(0, len(m), max(1, len(m) // 6)))
    ax.set_xticks(ticks)
    ax.set_xticklabels([m["ym_s"].iloc[i] for i in ticks], rotation=45, fontsize=8)

for ax in axes[n_ds:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Diurnal availability + AOD distribution ──
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Left: Diurnal availability (L2-level sources only)
src_hours = {"AERONET": df_aeronet["datetime"].dt.hour}
src_hours["VIIRS L2"] = df_viirs_l2["datetime"].dt.hour
if len(df_modis) > 0:
    src_hours["MODIS"] = df_modis["datetime"].dt.hour
if len(df_him_l2_valid) > 0:
    src_hours["Himawari L2"] = df_him_l2_valid["datetime"].dt.hour
if len(df_him_l3_valid) > 0:
    src_hours["Himawari L3"] = df_him_l3_valid["datetime"].dt.hour

w = 0.8 / len(src_hours)
color_list = ["#2196F3", "#4CAF50", "#9C27B0", "#FFD54F", "#FF9800"]
for i, (name, hrs) in enumerate(src_hours.items()):
    cnts = hrs.value_counts().reindex(range(24), fill_value=0)
    axes[0].bar(np.arange(24) + i * w, cnts.values, w, label=name, color=color_list[i % len(color_list)], alpha=0.8)
axes[0].set_xlabel("Hour (UTC)")
axes[0].set_ylabel("Records")
axes[0].set_title("Data availability by hour (UTC) — Local = UTC+7")
axes[0].set_xticks(range(24))
axes[0].legend(fontsize=8)

# Right: AOD distribution (box plot)
rows = []
for src, df, col in datasets_info:
    vals = df[col].dropna()
    vals = vals[(vals >= 0) & (vals <= 3)]
    for v in vals:
        rows.append({"source": src, "AOD": v})
df_box = pd.DataFrame(rows)

sns.boxplot(data=df_box, x="source", y="AOD", ax=axes[1],
            fliersize=2, showmeans=True, meanprops={"marker": "D", "markerfacecolor": "red", "ms": 6})
axes[1].set_title("AOD 550nm Distribution")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print("\nSummary statistics:")
print(df_box.groupby("source")["AOD"].describe().round(3).to_string())

## 4. L2-Level Validation: Satellite vs AERONET (±30 min time-matched)

Compares instantaneous L2 retrievals (VIIRS L2, Himawari L2, MODIS) against AERONET mean within ±30 min window.

In [ ]:
# ── 4.1 Scatter plots with regression + EE envelope (L2 group) ──
def plot_scatter_validation(matched_dict, title_prefix=""):
    """Plot scatter plots for satellite vs AERONET matched pairs."""
    valid = {k: v for k, v in matched_dict.items() if len(v) >= 3}
    n_plots = len(valid)
    if n_plots == 0:
        print("No valid matched pairs to plot.")
        return {}

    ncols = min(3, n_plots)
    nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
    axes = np.atleast_1d(axes).flatten()
    all_metrics = {}

    for ax, (src, df_m) in zip(axes, valid.items()):
        x, y = df_m["aer_aod"].values, df_m["sat_aod"].values
        m = compute_metrics(x, y)
        all_metrics[src] = m

        lim = max(x.max(), y.max()) * 1.15
        xl = np.linspace(0, lim, 100)

        # Expected Error envelope
        ax.fill_between(xl, xl - 0.05 - 0.15 * xl, xl + 0.05 + 0.15 * xl,
                         alpha=0.12, color="gray", label="EE (±0.05±15%)")
        ax.plot(xl, xl, "k--", lw=1, alpha=0.4, label="1:1")
        ax.scatter(x, y, s=15, alpha=0.5, color=COLORS.get(src.replace(" ", "_"), "gray"), edgecolors="none")

        # Regression line
        ax.plot(xl, m["Slope"] * xl + m["Intercept"], "r-", lw=1.5,
                label=f"y={m['Slope']:.2f}x{m['Intercept']:+.3f}")

        ax.set(xlabel="AERONET AOD 550nm", ylabel=f"{src} AOD 550nm",
               xlim=(0, lim), ylim=(0, lim), aspect="equal")
        ax.set_title(f"{title_prefix}{src} (N={m['N']})\n"
                     f"R={m['R']:.3f}  RMSE={m['RMSE']:.3f}  Bias={m['Bias']:+.3f}  EE={m['%EE']:.0f}%")
        ax.legend(fontsize=7, loc="upper left")

    for ax in axes[n_plots:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return all_metrics


l2_matched = {"VIIRS L2": matched_viirs_l2}
if len(matched_him_l2) > 0:
    l2_matched["Himawari L2"] = matched_him_l2
if len(matched_modis) > 0:
    l2_matched["MODIS"] = matched_modis

print("=" * 60)
print("L2-LEVEL VALIDATION (±30 min time-matched)")
print("=" * 60)
l2_metrics = plot_scatter_validation(l2_matched, title_prefix="L2: ")

if l2_metrics:
    print("\nL2 Metrics Summary:")
    print(pd.DataFrame(l2_metrics).T.round(4).to_string())

## 5. L3/D3-Level Validation: Satellite vs AERONET (Daily Aggregated)

Compares gridded/aggregated daily products (VIIRS D3, Himawari L3) against AERONET daily mean.
**No L2 data is mixed into this comparison.**

In [ ]:
# ── 5.1 Scatter plots (L3/D3 group) ──
l3_matched = {"VIIRS D3": matched_viirs_d3}
if len(matched_him_l3) > 0:
    l3_matched["Himawari L3"] = matched_him_l3

print("=" * 60)
print("L3/D3-LEVEL VALIDATION (Daily Aggregated)")
print("=" * 60)
l3_metrics = plot_scatter_validation(l3_matched, title_prefix="L3: ")

if l3_metrics:
    print("\nL3/D3 Metrics Summary:")
    print(pd.DataFrame(l3_metrics).T.round(4).to_string())

## 6. Time Series Comparison

In [ ]:
# ── 6.1 Build daily means for time series ──
def make_daily(df, col, src):
    d = df.copy()
    d["date_dt"] = pd.to_datetime(d["date"])
    daily = d.groupby("date_dt")[col].mean().reset_index()
    daily.columns = ["date", "aod"]
    daily["source"] = src
    return daily

daily_parts = {"AERONET": make_daily(df_aeronet, "AOD_550", "AERONET")}
daily_parts["VIIRS L2"] = make_daily(df_viirs_l2, "aod", "VIIRS L2")
daily_parts["VIIRS D3"] = make_daily(df_viirs_d3, "aod_mean", "VIIRS D3")
if len(df_modis) > 0:
    daily_parts["MODIS"] = make_daily(df_modis, "aod", "MODIS")
if len(df_him_l2_valid) > 0:
    daily_parts["Himawari L2"] = make_daily(df_him_l2_valid, "aod", "Himawari L2")
if len(df_him_l3_valid) > 0:
    daily_parts["Himawari L3"] = make_daily(df_him_l3_valid, "aod", "Himawari L3")

# ── L2 time series ──
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

l2_sources = ["AERONET", "VIIRS L2", "Himawari L2", "MODIS"]
l3_sources = ["AERONET", "VIIRS D3", "Himawari L3"]

for src in l2_sources:
    if src in daily_parts:
        d = daily_parts[src]
        axes[0].plot(d["date"], d["aod"], lw=0.8, alpha=0.6,
                     color=COLORS.get(src.replace(" ", "_"), "gray"), label=src)
axes[0].set_title("L2-Level Daily Mean AOD — VIIRS L2 + Himawari L2 + MODIS vs AERONET")
axes[0].set_ylabel("AOD 550nm")
axes[0].legend(ncol=4, fontsize=9)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

for src in l3_sources:
    if src in daily_parts:
        d = daily_parts[src]
        axes[1].plot(d["date"], d["aod"], lw=0.8, alpha=0.6,
                     color=COLORS.get(src.replace(" ", "_"), "gray"), label=src)
axes[1].set_title("L3/D3-Level Daily Mean AOD — VIIRS D3 + Himawari L3 vs AERONET")
axes[1].set_ylabel("AOD 550nm")
axes[1].legend(ncol=3, fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2 Rolling mean (30-day) — smoothed comparison ──
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

for src in l2_sources:
    if src in daily_parts:
        d = daily_parts[src].set_index("date")["aod"].sort_index()
        r = d.rolling(30, min_periods=10).mean()
        axes[0].plot(r.index, r.values, lw=2.5, color=COLORS.get(src.replace(" ", "_"), "gray"), label=src)
axes[0].set_title("30-Day Rolling Mean — L2 Group")
axes[0].set_ylabel("AOD 550nm")
axes[0].legend(ncol=4, fontsize=9)

for src in l3_sources:
    if src in daily_parts:
        d = daily_parts[src].set_index("date")["aod"].sort_index()
        r = d.rolling(30, min_periods=10).mean()
        axes[1].plot(r.index, r.values, lw=2.5, color=COLORS.get(src.replace(" ", "_"), "gray"), label=src)
axes[1].set_title("30-Day Rolling Mean — L3/D3 Group")
axes[1].set_ylabel("AOD 550nm")
axes[1].legend(ncol=3, fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

## 7. Density Scatter & Residual Analysis

In [ ]:
# ── 7.1 Density scatter (2D histogram) for datasets with many points ──
all_matched = {**{f"L2: {k}": v for k, v in l2_matched.items()},
               **{f"L3: {k}": v for k, v in l3_matched.items()}}
valid_matched = {k: v for k, v in all_matched.items() if len(v) >= 20}
n_plots = len(valid_matched)

if n_plots > 0:
    ncols = min(3, n_plots)
    nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for ax, (src, df_m) in zip(axes, valid_matched.items()):
        x, y = df_m["aer_aod"].values, df_m["sat_aod"].values
        lim = max(x.max(), y.max()) * 1.15
        h = ax.hist2d(x, y, bins=40, range=[[0, lim], [0, lim]], cmap="YlOrRd", cmin=1)
        plt.colorbar(h[3], ax=ax, label="Count")
        ax.plot([0, lim], [0, lim], "k--", lw=1, alpha=0.5)
        ax.set(xlabel="AERONET", ylabel="Satellite", title=f"Density: {src} (N={len(x)})", aspect="equal")

    for ax in axes[n_plots:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough matched pairs (need >=20) for density scatter.")

In [ ]:
# ── 7.2 Residual (Bias) analysis: satellite - AERONET vs AERONET AOD ──
if n_plots > 0:
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for ax, (src, df_m) in zip(axes, valid_matched.items()):
        x = df_m["aer_aod"].values
        residual = df_m["sat_aod"].values - x
        ax.scatter(x, residual, s=12, alpha=0.4, edgecolors="none")
        ax.axhline(0, color="k", ls="--", lw=1)
        # Running mean bias
        order = np.argsort(x)
        xs, rs = x[order], residual[order]
        win = max(len(xs) // 20, 5)
        rm = pd.Series(rs).rolling(win, min_periods=3, center=True).mean()
        ax.plot(xs, rm.values, "r-", lw=2, label="Running mean bias")
        ax.set(xlabel="AERONET AOD", ylabel="Satellite - AERONET", title=f"Residual: {src}")
        ax.legend(fontsize=8)

    for ax in axes[n_plots:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Seasonal Analysis

In [ ]:
# ── 8.1 Monthly mean AOD cycle — all sources on same axes (L2 and L3 separated) ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for src in l2_sources:
    if src in daily_parts:
        d = daily_parts[src].copy()
        d["month"] = d["date"].dt.month
        monthly = d.groupby("month")["aod"].agg(["mean", "std"]).reset_index()
        axes[0].errorbar(monthly["month"], monthly["mean"], yerr=monthly["std"],
                         fmt="o-", color=COLORS.get(src.replace(" ", "_"), "gray"),
                         label=src, capsize=3, lw=1.5, ms=5)
axes[0].set(xlabel="Month", ylabel="Mean AOD 550nm", title="Monthly AOD Cycle — L2 Group",
            xticks=range(1, 13))
axes[0].legend(fontsize=9)

for src in l3_sources:
    if src in daily_parts:
        d = daily_parts[src].copy()
        d["month"] = d["date"].dt.month
        monthly = d.groupby("month")["aod"].agg(["mean", "std"]).reset_index()
        axes[1].errorbar(monthly["month"], monthly["mean"], yerr=monthly["std"],
                         fmt="o-", color=COLORS.get(src.replace(" ", "_"), "gray"),
                         label=src, capsize=3, lw=1.5, ms=5)
axes[1].set(xlabel="Month", ylabel="Mean AOD 550nm", title="Monthly AOD Cycle — L3/D3 Group",
            xticks=range(1, 13))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 8.2 Seasonal scatter validation (L2 group, DJF/MAM/JJA/SON) ──
SEASONS = {1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
           6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON", 12: "DJF"}

# Collect seasonal metrics for all L2 sources
seasonal_metrics = []
for src, df_m in l2_matched.items():
    if len(df_m) < 10:
        continue
    df_m = df_m.copy()
    df_m["month"] = pd.to_datetime(df_m["datetime"]).dt.month
    df_m["season"] = df_m["month"].map(SEASONS)

    for season in ["DJF", "MAM", "JJA", "SON"]:
        sub = df_m[df_m["season"] == season]
        if len(sub) >= 3:
            m = compute_metrics(sub["aer_aod"].values, sub["sat_aod"].values)
            m["Source"] = src
            m["Season"] = season
            seasonal_metrics.append(m)

# Also for L3 sources
for src, df_m in l3_matched.items():
    if len(df_m) < 10:
        continue
    df_m = df_m.copy()
    df_m["month"] = pd.to_datetime(df_m["datetime"]).dt.month
    df_m["season"] = df_m["month"].map(SEASONS)

    for season in ["DJF", "MAM", "JJA", "SON"]:
        sub = df_m[df_m["season"] == season]
        if len(sub) >= 3:
            m = compute_metrics(sub["aer_aod"].values, sub["sat_aod"].values)
            m["Source"] = src
            m["Season"] = season
            seasonal_metrics.append(m)

if seasonal_metrics:
    df_seasonal = pd.DataFrame(seasonal_metrics)
    cols_order = ["Source", "Season", "N", "R", "R²", "RMSE", "MAE", "Bias", "%EE"]
    display_cols = [c for c in cols_order if c in df_seasonal.columns]
    print("Seasonal Validation Metrics:")
    print(df_seasonal[display_cols].round(3).to_string(index=False))

    # Bar chart: R² and RMSE by season
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for metric, ax, title in [("R²", axes[0], "R² by Season"), ("RMSE", axes[1], "RMSE by Season")]:
        pivot = df_seasonal.pivot_table(index="Season", columns="Source", values=metric)
        pivot = pivot.reindex(["DJF", "MAM", "JJA", "SON"])
        pivot.plot(kind="bar", ax=ax, rot=0)
        ax.set_title(title)
        ax.set_ylabel(metric)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough seasonal data for analysis.")

## 9. Correlation Heatmap (within-level only)

In [ ]:
# ── 9.1 Correlation heatmap — L2 and L3 groups separately ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# L2 group daily correlation
l2_daily = {src: daily_parts[src].set_index("date")["aod"]
            for src in l2_sources if src in daily_parts}
if len(l2_daily) >= 2:
    l2_wide = pd.DataFrame(l2_daily)
    corr_l2 = l2_wide.corr("pearson")
    sns.heatmap(corr_l2, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1,
                square=True, ax=axes[0])
    axes[0].set_title("Pearson r — L2 Group (daily)")
else:
    axes[0].text(0.5, 0.5, "Not enough L2 sources", ha="center", transform=axes[0].transAxes)

# L3 group daily correlation
l3_daily = {src: daily_parts[src].set_index("date")["aod"]
            for src in l3_sources if src in daily_parts}
if len(l3_daily) >= 2:
    l3_wide = pd.DataFrame(l3_daily)
    corr_l3 = l3_wide.corr("pearson")
    sns.heatmap(corr_l3, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1,
                square=True, ax=axes[1])
    axes[1].set_title("Pearson r — L3/D3 Group (daily)")
else:
    axes[1].text(0.5, 0.5, "Not enough L3 sources", ha="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

# Print co-located pair counts
print("\nCo-located daily pair counts:")
for group_name, group_daily in [("L2", l2_daily), ("L3", l3_daily)]:
    srcs = list(group_daily.keys())
    for i, s1 in enumerate(srcs):
        for s2 in srcs[i+1:]:
            n = pd.DataFrame({"a": group_daily[s1], "b": group_daily[s2]}).dropna().shape[0]
            print(f"  [{group_name}] {s1} vs {s2}: {n} days")

## 10. Combined Metrics Summary

In [ ]:
# ── 10.1 Combined validation metrics table ──
all_metrics_rows = []

for src, m in l2_metrics.items():
    row = {**m, "Source": src, "Level": "L2", "Matching": f"±{TIME_WINDOW_MIN}min"}
    all_metrics_rows.append(row)

for src, m in l3_metrics.items():
    row = {**m, "Source": src, "Level": "L3/D3", "Matching": "Daily mean"}
    all_metrics_rows.append(row)

if all_metrics_rows:
    df_metrics = pd.DataFrame(all_metrics_rows)
    cols = ["Source", "Level", "Matching", "N", "R", "R²", "RMSE", "MAE", "Bias", "Slope", "Intercept", "%EE"]
    display_cols = [c for c in cols if c in df_metrics.columns]
    print("=" * 80)
    print("COMBINED VALIDATION METRICS — Satellite vs AERONET (Nghia Do, Hanoi)")
    print("=" * 80)
    print(df_metrics[display_cols].round(4).to_string(index=False))
    print("\nEE = Expected Error envelope: ±(0.05 + 0.15 × AOD_AERONET)")
    print(f"L2 time matching: ±{TIME_WINDOW_MIN} min window (AERONET averaged)")
    print("L3/D3 matching: daily mean AERONET vs satellite daily product")
else:
    print("No metrics computed — check data availability.")

In [ ]:
# ── 10.2 Visual summary: bar chart of key metrics ──
if all_metrics_rows:
    df_m = pd.DataFrame(all_metrics_rows)
    df_m["label"] = df_m["Source"] + " (" + df_m["Level"] + ")"

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    metrics_to_plot = [("R²", "R² (higher is better)"), ("RMSE", "RMSE (lower is better)"),
                       ("Bias", "Mean Bias"), ("%EE", "% within EE")]

    for ax, (metric, title) in zip(axes, metrics_to_plot):
        if metric in df_m.columns:
            colors = [COLORS.get(s.replace(" ", "_"), "gray") for s in df_m["Source"]]
            bars = ax.barh(df_m["label"], df_m[metric], color=colors, edgecolor="gray", alpha=0.85)
            ax.set_title(title)
            ax.set_xlabel(metric)
            if metric == "Bias":
                ax.axvline(0, color="k", ls="--", lw=1)
            for bar, val in zip(bars, df_m[metric]):
                ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                        f" {val:.3f}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()